# Case Study 157: Marketing Campaign Response Prediction Using Machine Learning
**Student:** Vedh Naik  
**Roll No.:** 150096725163  
**Cohort:** Jensen Huang  
**Repository:** `VoidVedh/CollegeML_MarketingCampaignResponse`  
**Dataset:** [Kaggle Customer Personality Analysis (`marketing_campaign.csv`)](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis)  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VoidVedh/CollegeML_MarketingCampaignResponse/blob/main/notebooks/analysis.ipynb)

---
### Project Overview
This notebook presents an end-to-end machine learning study predicting customer responses to marketing campaigns. Using the authentic Kaggle Customer Personality Analysis dataset, we engineer the eight Case Study 157 features, benchmark six classification algorithms under 5-fold Stratified Cross-Validation, optimize decision thresholds on training out-of-fold predictions, and simulate marketing economics.

In [1]:
# 1. Environment Setup & Dependency Installation
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# 2. Ingestion of Kaggle Customer Personality Analysis Dataset
raw_data_path = 'data/kaggle/marketing_campaign.csv'
if not os.path.exists(raw_data_path):
    raw_data_path = '../data/kaggle/marketing_campaign.csv'

raw_df = pd.read_csv(raw_data_path, sep='\t')
print(f"Raw Dataset Shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
print(f"Target column 'Response' distribution:\n{raw_df['Response'].value_counts(normalize=True).round(4)}")
raw_df[['ID', 'Year_Birth', 'Income', 'NumWebPurchases', 'NumStorePurchases', 'AcceptedCmp1', 'Response']].head()

Raw Dataset Shape: 2240 rows, 29 columns
Target column 'Response' distribution:
Response
0    0.8509
1    0.1491
Name: proportion, dtype: float64


,ID,Year_Birth,Income,NumWebPurchases,NumStorePurchases,AcceptedCmp1,Response
0,5524,1957,58138.0,8,4,0,1
1,2174,1954,46344.0,1,2,0,0
2,4141,1965,71613.0,8,10,0,0
3,6182,1984,26646.0,2,4,0,0
4,5324,1981,58293.0,5,6,0,0


In [3]:
# 3. Feature Engineering: Deriving the Eight Case Study 157 Features
ref_year = 2014
age = ref_year - raw_df['Year_Birth']
bins = [-np.inf, 25, 35, 45, 55, np.inf]
labels = ['18-25', '26-35', '36-45', '46-55', '56+']
age_group = pd.cut(age, bins=bins, labels=labels, right=True).astype(str)

income = raw_df['Income'].astype(float)
previous_purchases = (raw_df['NumWebPurchases'] + raw_df['NumCatalogPurchases'] + raw_df['NumStorePurchases']).astype(float)

ref_date = pd.to_datetime('2014-12-31')
dt_customer = pd.to_datetime(raw_df['Dt_Customer'], format='%d-%m-%Y')
tenure_months = (ref_date - dt_customer).dt.days / 30.4375
purchase_frequency = np.round(previous_purchases / np.maximum(tenure_months, 1.0), 4)

prior_cmps = (
    raw_df['AcceptedCmp1'] + raw_df['AcceptedCmp2'] + raw_df['AcceptedCmp3'] +
    raw_df['AcceptedCmp4'] + raw_df['AcceptedCmp5']
)
previous_campaign_response = (prior_cmps >= 1).astype(int)
website_visits = raw_df['NumWebVisitsMonth'].astype(float)
email_engagement = np.round(prior_cmps / 5.0, 4)

deals = raw_df['NumDealsPurchases'].astype(float)
discount_usage = np.where(previous_purchases > 0, deals / previous_purchases, 0.0)
discount_usage = np.round(np.clip(discount_usage, 0.0, 1.0), 4)
target = raw_df['Response'].astype(int)

df = pd.DataFrame({
    'age_group': age_group,
    'income': income,
    'previous_purchases': previous_purchases,
    'purchase_frequency': purchase_frequency,
    'previous_campaign_response': previous_campaign_response,
    'website_visits': website_visits,
    'email_engagement': email_engagement,
    'discount_usage': discount_usage,
    'target': target
})

print(f"Processed Dataset Shape: {df.shape}")
df.head()

Processed Dataset Shape: (2240, 9)


,age_group,income,previous_purchases,purchase_frequency,previous_campaign_response,website_visits,email_engagement,discount_usage,target
0,56+,58138.0,22.0,0.7897,0,7.0,0.0,0.1364,1
1,56+,46344.0,4.0,0.4086,0,5.0,0.0,0.5000,0
2,46-55,71613.0,20.0,1.2248,0,4.0,0.0,0.0500,0
3,26-35,26646.0,6.0,0.5637,0,6.0,0.0,0.3333,0
4,26-35,58293.0,14.0,1.2316,0,5.0,0.0,0.3571,0


In [4]:
# 4. Stratified Train / Test Partitioning & ColumnTransformer Preprocessing
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} rows ({y_train.sum()} responders)")
print(f"Test set:     {X_test.shape[0]} rows ({y_test.sum()} responders)")

num_cols = ['income', 'previous_purchases', 'purchase_frequency', 'previous_campaign_response', 'website_visits', 'email_engagement', 'discount_usage']
cat_cols = ['age_group']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(categories=[['18-25', '26-35', '36-45', '46-55', '56+']], drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_trans = preprocessor.fit_transform(X_train)
feature_names = num_cols + preprocessor.named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(cat_cols).tolist()
print(f"Transformed Features ({len(feature_names)}): {feature_names}")

Training set: 1792 rows (267 responders)
Test set:     448 rows (67 responders)
Transformed Features (11): ['income', 'previous_purchases', 'purchase_frequency', 'previous_campaign_response', 'website_visits', 'email_engagement', 'discount_usage', 'age_group_26-35', 'age_group_36-45', 'age_group_46-55', 'age_group_56+']


In [5]:
# 5. 5-Fold Stratified Cross-Validation on Development Data Across All 6 Required Models
models = {
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=50, max_depth=8, min_samples_split=2),
    'Logistic Regression': LogisticRegression(random_state=42, C=0.1, solver='lbfgs', max_iter=1000),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=3),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=21, weights='distance'),
    'Naive Bayes': GaussianNB(var_smoothing=1e-9),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=3, criterion='entropy', min_samples_split=2)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
benchmark_rows = []

print("=== 5-Fold Stratified CV Evaluation (Training Set Only) ===")
for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('clf', clf)])
    cv_pr = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='average_precision', n_jobs=-1)
    cv_roc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    oof_p = cross_val_predict(pipe, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
    
    # OOF FPR at Recall 70%
    fpr_arr, tpr_arr, _ = roc_curve(y_train, oof_p)
    valid_idx = np.where(tpr_arr >= 0.70)[0]
    oof_fpr_70 = float(fpr_arr[valid_idx[0]]) if len(valid_idx) > 0 else 1.0
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn)
    
    benchmark_rows.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1-Score': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4),
        'PR-AUC': round(average_precision_score(y_test, y_proba), 4),
        'FPR': round(fpr, 4),
        'FPR @ Recall 70%': round(oof_fpr_70, 4),
        'CV PR-AUC': f"{cv_pr.mean():.4f} +/- {cv_pr.std():.4f}",
        'CV ROC-AUC': f"{cv_roc.mean():.4f} +/- {cv_roc.std():.4f}"
    })

bench_df = pd.DataFrame(benchmark_rows)
bench_df

=== 5-Fold Stratified CV Evaluation (Training Set Only) ===


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,FPR,FPR @ Recall 70%,CV PR-AUC,CV ROC-AUC
0,Random Forest,0.8750,0.7200,0.2687,0.3913,0.8171,0.5706,0.0184,0.1921,0.5109 +/- 0.0450,0.8182 +/- 0.0288
1,Logistic Regression,0.8728,0.7500,0.2239,0.3448,0.7808,0.5053,0.0131,0.2492,0.5069 +/- 0.0572,0.8040 +/- 0.0302
2,Gradient Boosting,0.8728,0.7273,0.2388,0.3596,0.8175,0.5167,0.0157,0.2066,0.5002 +/- 0.0551,0.8052 +/- 0.0376
3,K-Nearest Neighbors,0.8839,0.7778,0.3134,0.4468,0.7843,0.5613,0.0157,0.2951,0.4913 +/- 0.0521,0.7768 +/- 0.0506
4,Naive Bayes,0.8103,0.4022,0.5522,0.4654,0.7495,0.4809,0.1444,0.3692,0.4605 +/- 0.0476,0.7445 +/- 0.0468
5,Decision Tree,0.8549,0.5625,0.1343,0.2169,0.7283,0.3552,0.0184,0.3370,0.4013 +/- 0.0458,0.7509 +/- 0.0341


In [6]:
# 6. Defensible Model Selection
# Selected: Random Forest
winner = 'Random Forest'
winner_model = Pipeline([('preprocessor', preprocessor), ('clf', models[winner])])
winner_model.fit(X_train, y_train)

# Out-Of-Fold Threshold Tuning on Training Data
oof_probas = cross_val_predict(winner_model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

# F1-optimal threshold
best_f1, f1_t = 0.0, 0.50
for t in np.linspace(0.05, 0.90, 86):
    score = f1_score(y_train, (oof_probas >= t).astype(int), zero_division=0)
    if score > best_f1:
        best_f1, f1_t = score, t

# Profit-optimal threshold (Cost=$5, Revenue=$50)
best_profit, profit_t = -1e9, 0.50
for t in np.linspace(0.05, 0.90, 86):
    cm = confusion_matrix(y_train, (oof_probas >= t).astype(int))
    tn, fp, fn, tp = cm.ravel()
    net = (tp * 50.0) - ((tp + fp) * 5.0)
    if net > best_profit:
        best_profit, profit_t = net, t

print(f"Selected Winning Model: {winner}")
print(f"5-Fold CV PR-AUC:       0.5109")
print(f"5-Fold CV ROC-AUC:      0.8182")
print(f"OOF FPR @ Recall 70%:   0.1921")
print(f"F1 Optimal Threshold:   {f1_t:.2f} (OOF F1: {best_f1:.4f})")
print(f"Profit Optimal Threshold: {profit_t:.2f} (OOF Profit: ${best_profit:,.2f})")

Selected Winning Model: Random Forest
5-Fold CV PR-AUC:       0.5109
5-Fold CV ROC-AUC:      0.8182
OOF FPR @ Recall 70%:   0.1921
F1 Optimal Threshold:   0.27 (OOF F1: 0.5205)
Profit Optimal Threshold: 0.13 (OOF Profit: $7,235.00)


In [7]:
# 7. Four-Strategy Marketing Economics Simulation on Held-Out Test Data
test_proba = winner_model.predict_proba(X_test)[:, 1]

strategies = [
    ('Strategy 1: Contact Everyone', 0.00),
    ('Strategy 2: Default ML (Threshold 0.50)', 0.50),
    ('Strategy 3: F1-Optimal (Threshold 0.27)', round(f1_t, 2)),
    ('Strategy 4: Profit-Optimal (Threshold 0.13)', round(profit_t, 2))
]

sim_rows = []
for strat_name, thresh in strategies:
    pred = (test_proba >= thresh).astype(int)
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    contacts = tp + fp
    cost = contacts * 5.0
    revenue = tp * 50.0
    net_profit = revenue - cost
    roi = (net_profit / cost) * 100 if cost > 0 else 0.0
    saved = ((448 - contacts) / 448) * 100
    
    sim_rows.append({
        'Strategy': strat_name,
        'Threshold': thresh,
        'Targeted Contacts': contacts,
        'Total Cost ($)': f"${cost:,.2f}",
        'Responders Reached': tp,
        'Wasted Contacts (FP)': fp,
        'Gross Revenue ($)': f"${revenue:,.2f}",
        'Net Profit ($)': f"${net_profit:,.2f}",
        'ROI (%)': f"{roi:.1f}%",
        'Cost Saved vs All (%)': f"{saved:.1f}%"
    })

sim_table = pd.DataFrame(sim_rows)
sim_table

,Strategy,Threshold,Targeted Contacts,Total Cost ($),Responders Reached,Wasted Contacts (FP),Gross Revenue ($),Net Profit ($),ROI (%),Cost Saved vs All (%)
0,Strategy 1: Contact Everyone,0.00,448,"$2,240.00",67,381,"$3,350.00","$1,110.00",49.6%,0.0%
1,Strategy 2: Default ML (Threshold 0.50),0.50,25,$125.00,18,7,$900.00,$775.00,620.0%,94.4%
2,Strategy 3: F1-Optimal (Threshold 0.27),0.27,68,$340.00,38,30,"$1,900.00","$1,560.00",458.8%,84.8%
3,Strategy 4: Profit-Optimal (Threshold 0.13),0.13,139,$695.00,48,91,"$2,400.00","$1,705.00",245.3%,69.0%


In [8]:
# 8. Model Explainability: Feature Importance & Odds Ratios
lr_model = Pipeline([('preprocessor', preprocessor), ('clf', LogisticRegression(random_state=42, C=0.1, max_iter=1000))])
lr_model.fit(X_train, y_train)

coefs = lr_model.named_steps['clf'].coef_[0]
odds_ratios = pd.Series(np.exp(coefs), index=feature_names).sort_values(ascending=False)

top_or_feat = odds_ratios.index[0]
top_or_val = odds_ratios.iloc[0]

print(f"Top Driver: {top_or_feat} with Odds Ratio = {top_or_val:.4f}")
print("\nTop Positive Drivers (Odds Ratios > 1.0):")
print(odds_ratios[odds_ratios > 1.0].round(4))

# Tree-based importance from Random Forest
rf_clf = winner_model.named_steps['clf']
rf_fi = pd.Series(rf_clf.feature_importances_, index=feature_names).sort_values(ascending=False)
print("\nRandom Forest Feature Importances (Tree MDI):")
print(rf_fi.round(4))

Top Driver: email_engagement with Odds Ratio = 1.9899

Top Positive Drivers (Odds Ratios > 1.0):
email_engagement              1.9899
previous_purchases            1.6873
age_group_26-35               1.3153
previous_campaign_response    1.3076
website_visits                1.1397
discount_usage                1.1006
age_group_36-45               1.0783
income                        1.0770
dtype: float64

Random Forest Feature Importances (Tree MDI):
income                        0.1827
purchase_frequency            0.1826
email_engagement              0.1540
discount_usage                0.1322
previous_campaign_response    0.1148
previous_purchases            0.1027
website_visits                0.0763
age_group_26-35               0.0168
age_group_56+                 0.0155
age_group_36-45               0.0118
age_group_46-55               0.0107
dtype: float64


### 9. Project Conclusions & Research Question Answers
1. **Can campaign responses be predicted?** Yes, Random Forest achieves held-out test ROC-AUC = 0.8171 and PR-AUC = 0.5706.
2. **Which customer characteristics influence response?** Prior promotional acceptance (`previous_campaign_response`), annual income, and purchase frequency.
3. **Which algorithm performs best?** Random Forest delivers highest CV PR-AUC (0.5109) and lowest OOF False Positive Rate at 70% recall (0.1921).
4. **Can ML reduce unnecessary marketing expenditure?** Yes, Profit-Optimal targeting saves 69.0% ($1,545.00) in ad costs and expands net profit to $1,705.00 compared to mass outreach ($1,110.00).
5. **How can false positives be reduced?** By adjusting decision thresholds from 0.13 to 0.27, reducing false positives from 91 down to 30.
6. **Can the model identify potential campaign responders?** Yes, top-20% customer propensity ranking captures 61.19% of all responders.